# RT-DETR Knowledge Distillation — Ablation Analysis

CS229 Project — Umut Onur Yasar

This notebook:
1. Loads mAP and FPS results from all 12 ablation runs.
2. Produces a formatted results table.
3. Generates a mAP-FPS Pareto front plot.
4. Performs basic statistical analysis.


In [ ]:
import os
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

RUNS_ROOT = Path('../runs')  # adjust if needed

In [ ]:
def parse_map_from_log(log_path: Path) -> float | None:
    """Extract primary mAP@[.5:.95] from eval.log."""
    if not log_path.exists():
        return None
    text = log_path.read_text()
    # Look for: AP@[.5:.95]  : 0.XXXX
    m = re.search(r'AP@\[.5:.95\]\s*:\s*([0-9.]+)', text)
    if m:
        return float(m.group(1))
    # Fallback: look for pycocotools summary line
    m = re.search(r'Average Precision.*all.*=\s*([0-9.]+)', text)
    return float(m.group(1)) if m else None


def parse_fps_from_log(log_path: Path) -> tuple[float | None, float | None]:
    """Extract mean FPS ± std from fps.log."""
    if not log_path.exists():
        return None, None
    text = log_path.read_text()
    m = re.search(r'Mean FPS\s*:\s*([0-9.]+)\s*±\s*([0-9.]+)', text)
    if m:
        return float(m.group(1)), float(m.group(2))
    return None, None


# Define ablation metadata
RUN_META = [
    dict(run_id=0,  tag='run00_baseline',       kd_type='None',    lam=0.0, T='-',  expected_map=48.9),
    dict(run_id=1,  tag='run01_logit_l0.5_t2',  kd_type='Logit',   lam=0.5, T=2,   expected_map=49.2),
    dict(run_id=2,  tag='run02_logit_l0.5_t4',  kd_type='Logit',   lam=0.5, T=4,   expected_map=49.4),
    dict(run_id=3,  tag='run03_logit_l0.5_t8',  kd_type='Logit',   lam=0.5, T=8,   expected_map=49.2),
    dict(run_id=4,  tag='run04_logit_l1.0_t2',  kd_type='Logit',   lam=1.0, T=2,   expected_map=49.3),
    dict(run_id=5,  tag='run05_logit_l1.0_t4',  kd_type='Logit',   lam=1.0, T=4,   expected_map=49.6),
    dict(run_id=6,  tag='run06_logit_l1.0_t8',  kd_type='Logit',   lam=1.0, T=8,   expected_map=49.4),
    dict(run_id=7,  tag='run07_feature_l0.5',   kd_type='Feature', lam=0.5, T='-', expected_map=50.2),
    dict(run_id=8,  tag='run08_feature_l1.0',   kd_type='Feature', lam=1.0, T='-', expected_map=50.4),
    dict(run_id=9,  tag='run09_extended',        kd_type='Feature', lam=1.0, T='-', expected_map=50.8),
    dict(run_id=10, tag='run10_extended',        kd_type='Feature', lam=1.0, T='-', expected_map=None),
    dict(run_id=11, tag='run11_extended',        kd_type='Feature', lam=1.0, T='-', expected_map=None),
]

# Load actual results
rows = []
for meta in RUN_META:
    run_dir = RUNS_ROOT / meta['tag']
    actual_map = parse_map_from_log(run_dir / 'eval.log')
    fps_mean, fps_std = parse_fps_from_log(run_dir / 'fps.log')
    rows.append({
        'Run': meta['run_id'],
        'Tag': meta['tag'],
        'KD Type': meta['kd_type'],
        'λ': meta['lam'],
        'T': meta['T'],
        'Expected mAP': meta['expected_map'],
        'Actual mAP': actual_map,
        'FPS (mean)': fps_mean,
        'FPS (std)': fps_std,
    })

df = pd.DataFrame(rows)
print(f"Loaded {df['Actual mAP'].notna().sum()} completed runs out of {len(df)} total.")

In [ ]:
# ----- Results Table -----
display_df = df[[
    'Run', 'KD Type', 'λ', 'T',
    'Expected mAP', 'Actual mAP', 'FPS (mean)', 'FPS (std)'
]].copy()

# Format numbers
display_df['Expected mAP'] = display_df['Expected mAP'].map(lambda x: f'{x:.1f}' if x else '—')
display_df['Actual mAP']   = display_df['Actual mAP'].map(lambda x: f'{x:.2f}' if x else '—')
display_df['FPS (mean)']   = display_df['FPS (mean)'].map(lambda x: f'{x:.1f}' if x else '—')
display_df['FPS (std)']    = display_df['FPS (std)'].map(lambda x: f'{x:.1f}' if x else '—')

print('=' * 80)
print('RT-DETR Knowledge Distillation — Ablation Results')
print('=' * 80)
print(display_df.to_string(index=False))
print('=' * 80)

In [ ]:
# ----- mAP-FPS Pareto Plot -----
fig, ax = plt.subplots(figsize=(9, 6))

completed = df[df['Actual mAP'].notna() & df['FPS (mean)'].notna()].copy()

# Color by KD type
color_map = {'None': '#888888', 'Logit': '#2196F3', 'Feature': '#E91E63'}
marker_map = {'None': 's', 'Logit': 'o', 'Feature': '^'}

for _, row in completed.iterrows():
    kd = row['KD Type']
    ax.scatter(
        row['FPS (mean)'], row['Actual mAP'],
        color=color_map.get(kd, 'black'),
        marker=marker_map.get(kd, 'o'),
        s=120, zorder=3,
        edgecolors='white', linewidths=0.8
    )
    label = f"R{int(row['Run'])}"
    ax.annotate(
        label,
        (row['FPS (mean)'], row['Actual mAP']),
        textcoords='offset points', xytext=(6, 4),
        fontsize=8.5, color='#333333'
    )

# Pareto frontier
if len(completed) > 1:
    pts = completed[['FPS (mean)', 'Actual mAP']].values
    pareto_mask = np.ones(len(pts), dtype=bool)
    for i, (fps_i, map_i) in enumerate(pts):
        for j, (fps_j, map_j) in enumerate(pts):
            if i != j and fps_j >= fps_i and map_j >= map_i and (fps_j > fps_i or map_j > map_i):
                pareto_mask[i] = False
                break
    pareto_pts = pts[pareto_mask]
    pareto_pts = pareto_pts[pareto_pts[:, 0].argsort()]
    ax.plot(pareto_pts[:, 0], pareto_pts[:, 1], 'k--', alpha=0.4, linewidth=1.2, label='Pareto front')

# Teacher reference line
ax.axhline(53.1, color='#FF9800', linestyle=':', linewidth=1.5, alpha=0.8, label='Teacher (RT-DETR-L) 53.1')
ax.axhline(48.9, color='#888888', linestyle=':', linewidth=1.5, alpha=0.6, label='Baseline (no KD) 48.9')

# Legend
legend_handles = [
    mpatches.Patch(color='#888888', label='Baseline'),
    mpatches.Patch(color='#2196F3', label='Logit-KD'),
    mpatches.Patch(color='#E91E63', label='Feature-KD'),
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9)

ax.set_xlabel('FPS (batch=1, RTX 3050, fp32)', fontsize=11)
ax.set_ylabel('mAP@[.5:.95]', fontsize=11)
ax.set_title('RT-DETR Student: mAP vs. FPS Trade-off\n(RT-DETR-S w/ Knowledge Distillation)', fontsize=12)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('../results_pareto.pdf', bbox_inches='tight')
plt.savefig('../results_pareto.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: results_pareto.pdf / .png')

In [ ]:
# ----- Grouped bar chart: mAP by KD type and lambda -----
completed_real = df[df['Actual mAP'].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Logit-KD — mAP vs Temperature, grouped by lambda
logit = completed_real[completed_real['KD Type'] == 'Logit']
if len(logit) > 0:
    for lam, grp in logit.groupby('λ'):
        grp_sorted = grp.sort_values('T')
        axes[0].plot(
            grp_sorted['T'].astype(str),
            grp_sorted['Actual mAP'],
            marker='o', label=f'λ={lam}'
        )
axes[0].axhline(
    completed_real[completed_real['KD Type'] == 'None']['Actual mAP'].values[0]
    if len(completed_real[completed_real['KD Type'] == 'None']) > 0 else 48.9,
    color='gray', linestyle='--', label='Baseline'
)
axes[0].set_xlabel('Temperature T')
axes[0].set_ylabel('mAP@[.5:.95]')
axes[0].set_title('Logit-KD: mAP vs Temperature')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: Feature-KD vs Logit-KD best per lambda
summary = []
baseline_map = completed_real[completed_real['KD Type'] == 'None']['Actual mAP']
baseline_map = baseline_map.values[0] if len(baseline_map) > 0 else 48.9
summary.append({'Config': 'Baseline', 'mAP': baseline_map})

for kd_type in ['Logit', 'Feature']:
    subset = completed_real[completed_real['KD Type'] == kd_type]
    for lam in [0.5, 1.0]:
        sub2 = subset[subset['λ'] == lam]
        if len(sub2) > 0:
            best_map = sub2['Actual mAP'].max()
            summary.append({'Config': f'{kd_type}\nλ={lam}', 'mAP': best_map})

configs = [r['Config'] for r in summary]
maps = [r['mAP'] for r in summary]
colors = ['#888888'] + ['#2196F3'] * 2 + ['#E91E63'] * 2
bars = axes[1].bar(range(len(configs)), maps, color=colors[:len(configs)], width=0.6)
axes[1].set_xticks(range(len(configs)))
axes[1].set_xticklabels(configs, fontsize=9)
axes[1].set_ylabel('Best mAP@[.5:.95]')
axes[1].set_title('Best mAP by KD Type and λ')
axes[1].set_ylim(min(maps) - 0.5 if maps else 48, max(maps) + 0.5 if maps else 51)
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, maps):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('../results_bar.pdf', bbox_inches='tight')
plt.savefig('../results_bar.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ----- Statistical Summary -----
print('=== KD Type Comparison ===')
for kd_type in ['None', 'Logit', 'Feature']:
    subset = df[df['KD Type'] == kd_type]['Actual mAP'].dropna()
    if len(subset) > 0:
        print(f'{kd_type:10s}: n={len(subset):2d}  '
              f'mean={subset.mean():.3f}  '
              f'std={subset.std():.3f}  '
              f'max={subset.max():.3f}')

print()
baseline_map = df[df['KD Type']=='None']['Actual mAP'].values
baseline_map = baseline_map[0] if len(baseline_map) > 0 else 48.9

best_logit = df[df['KD Type']=='Logit']['Actual mAP'].max() if len(df[df['KD Type']=='Logit']['Actual mAP'].dropna()) > 0 else None
best_feature = df[df['KD Type']=='Feature']['Actual mAP'].max() if len(df[df['KD Type']=='Feature']['Actual mAP'].dropna()) > 0 else None

print(f'Baseline mAP:          {baseline_map:.3f}')
if best_logit:
    print(f'Best Logit-KD mAP:     {best_logit:.3f} (+{best_logit-baseline_map:.3f})')
if best_feature:
    print(f'Best Feature-KD mAP:   {best_feature:.3f} (+{best_feature-baseline_map:.3f})')
if best_logit and best_feature:
    print(f'Feature vs Logit delta: +{best_feature-best_logit:.3f}')